# EDA 08 — Connectivite & Teletravail (ARCEP + INSEE)
**Sources** : ARCEP Mobile | ARCEP Fibre | INSEE RP2020 Logements

In [ ]:

import os, glob, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings("ignore")

PALETTE = {
    "primary":   "#1565C0",
    "secondary": "#E53935",
    "accent":    "#2E7D32",
    "neutral":   "#546E7A",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
})

BRONZE_ROOT = os.path.abspath("../../data/bronze")
NB_DIR      = os.path.abspath(".")
FIGURES_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(name, source_prefix, tight=True):
    dest = os.path.join(FIGURES_DIR, source_prefix)
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{name}.png")
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> figures/{source_prefix}/{name}.png")

print("Setup OK — figures ->", FIGURES_DIR)


In [ ]:

def draw_schema(title, columns, source_prefix, filename="schema"):
    n_rows = len(columns)
    fig_h  = max(3.0, 0.48 * n_rows + 1.6)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.99, ha="center")
    headers = ["Colonne", "Type", "Description"]
    col_x   = [0.0, 0.22, 0.37]
    col_w   = [0.22, 0.15, 0.63]
    row_h   = 1.0 / (n_rows + 1)
    for i, (hdr, x, w) in enumerate(zip(headers, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, 1 - row_h), w - 0.006, row_h * 0.88,
            boxstyle="round,pad=0.01", linewidth=0.5,
            edgecolor="#90A4AE", facecolor="#1565C0",
            transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, 1 - row_h/2, hdr,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
    type_colors = {
        "str":"#1565C0","int":"#2E7D32","float":"#6A1B9A",
        "datetime":"#E65100","bool":"#AD1457","date":"#00695C",
    }
    for ridx, (col_name, col_type, col_desc) in enumerate(columns):
        y_top = 1 - (ridx + 2) * row_h
        bg = "#F5F7FA" if ridx % 2 == 0 else "white"
        for x, w in zip(col_x, col_w):
            rect = mpatches.FancyBboxPatch(
                (x, y_top), w - 0.006, row_h * 0.88,
                boxstyle="round,pad=0.01", linewidth=0.5,
                edgecolor="#CFD8DC", facecolor=bg,
                transform=ax.transAxes, clip_on=False)
            ax.add_patch(rect)
        y_mid = y_top + row_h * 0.44
        tc = type_colors.get(col_type.split("[")[0].lower(), "#37474F")
        ax.text(col_x[0]+col_w[0]/2, y_mid, col_name,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, fontweight="bold", color="#1A237E")
        ax.text(col_x[1]+col_w[1]/2, y_mid, col_type,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, fontfamily="monospace", color=tc)
        ax.text(col_x[2]+0.01, y_mid, col_desc,
                transform=ax.transAxes, ha="left", va="center",
                fontsize=9.5, color="#37474F")
    plt.subplots_adjust(top=0.92, bottom=0)
    save_fig(filename, source_prefix)
    plt.show()


In [ ]:
PREFIX = "08_connectivity"
draw_schema(
    "Bronze Schema — ARCEP Mobile (couverture 4G/5G par commune x operateur)",
    [
        ("commune_code",  "str",   "Code INSEE commune (751xx)"),
        ("arrondissement","int",   "Numero arrondissement (1-20)"),
        ("operateur",     "str",   "Operateur : orange | sfr | bouygues | free"),
        ("has_4g",        "bool",  "Couverture 4G disponible sur la commune"),
        ("has_5g",        "bool",  "Couverture 5G disponible sur la commune"),
        ("pct_pop_4g",    "float", "% population couverte en 4G"),
        ("pct_pop_5g",    "float", "% population couverte en 5G"),
        ("periode",       "str",   "Trimestre de reference (ex: 2024-T3)"),
        ("ingested_at",   "datetime","Horodatage UTC d'ingestion"),
    ], PREFIX, "schema_arcep_mobile"
)

In [ ]:
draw_schema(
    "Bronze Schema — ARCEP Fibre FttH",
    [
        ("commune_code",      "str",   "Code INSEE commune"),
        ("arrondissement",    "int",   "Numero arrondissement"),
        ("nb_local_ftth",     "int",   "Locaux eligibles FttH (fibre jusqu'au logement)"),
        ("nb_local_total",    "int",   "Total locaux dans la commune"),
        ("pct_eligible_ftth", "float", "Taux d'eligibilite FttH (%)"),
        ("trimestre",         "str",   "Trimestre de reference"),
        ("ingested_at",       "datetime","Horodatage UTC d'ingestion"),
    ], PREFIX, "schema_arcep_fibre"
)

In [ ]:
draw_schema(
    "Bronze Schema — INSEE RP2020 Logements par taille",
    [
        ("commune_code",       "str",   "Code INSEE commune"),
        ("arrondissement",     "int",   "Numero arrondissement"),
        ("nb_logements_total", "int",   "Total de logements dans la commune"),
        ("nb_t1",              "int",   "Logements T1 (studio)"),
        ("nb_t2",              "int",   "Logements T2"),
        ("nb_t3",              "int",   "Logements T3"),
        ("nb_t4",              "int",   "Logements T4"),
        ("nb_t5_plus",         "int",   "Logements T5 et plus"),
        ("pct_t2_t3",          "float", "Proportion T2+T3 — proxy 'logements teleravail-friendly'"),
        ("annee_ref",          "int",   "Annee de reference RP (2020)"),
        ("ingested_at",        "datetime","Horodatage UTC d'ingestion"),
    ], PREFIX, "schema_insee_logements"
)

In [ ]:
def load_latest(pattern):
    files = sorted(glob.glob(pattern, recursive=True))
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True) if files else pd.DataFrame()
df_mobile = load_latest(f"{BRONZE_ROOT}/arcep_mobile/**/*.parquet")
df_fibre  = load_latest(f"{BRONZE_ROOT}/arcep_fibre/**/*.parquet")
df_log    = load_latest(f"{BRONZE_ROOT}/insee_logements/**/*.parquet")
rng = np.random.default_rng(42)
if df_mobile.empty:
    rows = []
    for a in range(1,21):
        for op in ["orange","sfr","bouygues","free"]:
            rows.append({"commune_code":f"751{a:02d}","arrondissement":a,"operateur":op,
                "has_4g":True,"has_5g":a<=12,"pct_pop_4g":min(100,max(75,rng.normal(95,5))),
                "pct_pop_5g":min(100,max(0,rng.normal(65,15))) if a<=12 else 0.0,"periode":"2024-T3"})
    df_mobile = pd.DataFrame(rows)
if df_fibre.empty:
    df_fibre = pd.DataFrame([{"commune_code":f"751{a:02d}","arrondissement":a,
        "pct_eligible_ftth":min(100,max(70,rng.normal(90,8))),"trimestre":"2024-T3"} for a in range(1,21)])
if df_log.empty:
    df_log = pd.DataFrame([{"commune_code":f"751{a:02d}","arrondissement":a,
        "nb_logements_total":int(rng.uniform(10000,60000)),"nb_t1":int(rng.uniform(1000,8000)),
        "nb_t2":int(rng.uniform(3000,15000)),"nb_t3":int(rng.uniform(2000,12000)),
        "nb_t4":int(rng.uniform(1000,8000)),"nb_t5_plus":int(rng.uniform(500,4000)),
        "pct_t2_t3":rng.uniform(35,55),"annee_ref":2020} for a in range(1,21)])
print(f"Mobile {df_mobile.shape} | Fibre {df_fibre.shape} | Logements {df_log.shape}")

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(16,6))
op_cov = df_mobile.groupby("operateur")["pct_pop_4g"].mean().sort_values(ascending=False)
op_cols = ["#FF6F00","#E53935","#00897B","#1E88E5"][:len(op_cov)]
bars = axes[0].bar(op_cov.index, op_cov.values, color=op_cols, edgecolor="white", width=0.6)
axes[0].set_ylim(0,105); axes[0].set_ylabel("Couverture 4G moyenne (%)"); axes[0].set_title("Couverture 4G par operateur — Paris")
for bar, val in zip(bars, op_cov.values):
    axes[0].text(bar.get_x()+bar.get_width()/2,val+0.5,f"{val:.1f}%",ha="center",fontsize=11,fontweight="bold")
arr_4g = df_mobile.groupby("arrondissement")["pct_pop_4g"].mean().sort_values()
axes[1].barh(arr_4g.index.astype(str),arr_4g.values,color=plt.cm.RdYlGn(arr_4g.values/100),edgecolor="white")
axes[1].set_xlabel("Couverture 4G moyenne (%)"); axes[1].set_title("Couverture 4G par arrondissement (moy. 4 operateurs)"); axes[1].set_xlim(0,105)
save_fig("couverture_4g", PREFIX); plt.show()

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(16,6))
df_f_s = df_fibre.sort_values("pct_eligible_ftth",ascending=False)
norm = (df_f_s["pct_eligible_ftth"]-df_f_s["pct_eligible_ftth"].min())/(df_f_s["pct_eligible_ftth"].max()-df_f_s["pct_eligible_ftth"].min())
axes[0].bar(df_f_s["arrondissement"].astype(str),df_f_s["pct_eligible_ftth"],color=plt.cm.Blues(0.3+norm*0.65),edgecolor="white")
axes[0].set_xlabel("Arrondissement"); axes[0].set_ylabel("% locaux eligibles FttH"); axes[0].set_title("Taux d'eligibilite fibre optique (FttH)"); axes[0].set_ylim(0,105)
plt.setp(axes[0].xaxis.get_majorticklabels(),rotation=45)
df_l_s = df_log.sort_values("arrondissement")
types = ["nb_t1","nb_t2","nb_t3","nb_t4","nb_t5_plus"]; t_cols = ["#EF9A9A","#90CAF9","#A5D6A7","#FFE082","#CE93D8"]
bot = np.zeros(20)
for col, c2, lab in zip(types,t_cols,["T1","T2","T3","T4","T5+"]):
    axes[1].bar(df_l_s["arrondissement"].astype(str),df_l_s[col],bottom=bot,label=lab,color=c2,edgecolor="white",linewidth=0.4)
    bot += df_l_s[col].values
axes[1].set_xlabel("Arrondissement"); axes[1].set_ylabel("Logements"); axes[1].set_title("Repartition logements par taille (T1 -> T5+)")
axes[1].legend(loc="upper right"); plt.setp(axes[1].xaxis.get_majorticklabels(),rotation=45)
save_fig("fibre_et_logements", PREFIX); plt.show()